## Homework: Vector Search

### 1. Setup Environment

### 1.1. Fetches helper scripts from the embed/ directory of the course repo (run once time)

1.1.1. Set ENV to URL target

In [5]:
#PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"


#### 1.1.2. Download all scripts

In [6]:
#!wget $PREFIX/download.py
#!wget $PREFIX/embedder.py

### 2. Loading the data from Git

#### 2.1. Fetch data from course repository

In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [8]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

### 2.2. Combine context and filename for each document

In [9]:
texts = [doc["content"] + " " + doc["filename"] for doc in documents]

In [10]:
texts[0]

'# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language 

### 2.3. Call embedder an text

In [11]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

2026-06-27 15:05:10.327559581 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [12]:
v1.dot(dv), v2.dot(dv)

(np.float64(0.3233238425677314), np.float64(0.01973045838992233))

### 2.4. Q1 - Query Embedding

In [13]:
qw1 = "How does approximate nearest neighbor search work?"

In [14]:
vqw1 = embed.encode(qw1)

In [15]:
vqw1[0]

np.float64(-0.02058203437252893)

### 2.5. Embed in batches

In [16]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
data_emb = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    data_emb.extend(batch_vectors)

data_emb = np.array(data_emb)

  0%|          | 0/2 [00:00<?, ?it/s]

### 3. Search and Q2. Cosine similarity

In [17]:
qw2 = "02-vector-search/lessons/07-sqlitesearch-vector.md?"
v_query = embed.encode(qw2)

scores = data_emb.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [18]:
vqw2 = embed.encode(documents[idx]['content'])

In [19]:
vqw2.dot(vqw1)

np.float64(0.36107027225589694)

### 3. Q3. Chunking and search by hand

#### 3.1. Chunk the pages the same way as in homework 1

In [20]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [21]:
texts_chunks = [c["content"] for c in chunks]
data_chunks = np.array(embed.encode_batch(texts_chunks))   


In [22]:
scores = data_chunks.dot(vqw1)

In [23]:
# 6. Chunk de maior score
best = int(np.argmax(scores))
print("score:", scores[best])
print("filename:", chunks[best]["filename"])

score: 0.6489017718578813
filename: 02-vector-search/lessons/07-sqlitesearch-vector.md


### 4. Q4. Vector search with minsearch

In [36]:
from minsearch import VectorSearch

vindex_q4 = VectorSearch()
vindex_q4.fit(data_emb, documents)

In [25]:
query_q4 = "What metric do we use to evaluate a search engine?"

In [26]:
query_vector_q4 = embed.encode(query_q4)

In [27]:
results_q4 = vindex_q4.search(query_vector_q4, num_results=5)

In [28]:
results_q4[0]

{'content': '# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup, each query 

### 5. Q5. Vector search with minsearch

In [38]:
from minsearch import Index

text_index_q5 = Index(text_fields=["content"], keyword_fields=[])
text_index_q5.fit(documents)        # mesma lista de chunks (dicts com 'content')

In [30]:
query_q5 = "How do I store vectors in PostgreSQL?"

In [39]:
results_q5 = text_index_q5.search(query_q5, num_results=5)
results_q5[0]

{'content': '# Embeddings\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=kJOlW1HeMp4&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nBefore we can do vector search, we need to turn our text into vectors.\nWe call this process embedding: we embed text into a vector space. The\nvectors we get back are also called "embeddings."\n\n## Word embeddings and sentence embeddings\n\nThis idea comes from\n[word2vec](https://en.wikipedia.org/wiki/Word2vec). The model learns to\nplace words as points in a multi-dimensional space. Words with similar\nmeanings land close to each other.\n\nImagine a 2D space where "enroll" and "join" are near each other and\n"Docker" is far away:\n\n```text\n        · enroll\n       · join\n                   · Docker\n```\n\nThe same idea works for entire sentences:\n\n```text\nQ1: "I just discovered the course. Can I still join it?"\nQ2: "I just found out about the program. Can I still enroll?"\n\nThese two are close - they mean the same thing.\n\nQ3: "Ho

### 6. Q6. Vector search with minsearch

In [44]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["content"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [45]:
query_q6 = "How do I give the model access to tools?"

# os dois buscadores que você já montou nos Q4 e Q5
text_results   = text_index_q5.search(query_q6, num_results=5) 
vector_results = vindex_q4.search(embed.encode(query_q6), num_results=5)


In [46]:
print(text_results[0].keys())

dict_keys(['content', 'filename'])


In [47]:
fused = rrf([text_results, vector_results], k=60, num_results=5)
print(fused[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
